# 08: QAT with onebitllms -> smelt inference

Train a ternary model with [onebitllms](https://github.com/tiiuae/onebitllms)
(STE + Triton kernels), then pack for fast CPU inference with smelt.

| step | tool | what |
|:---|:---|:---|
| replace linears | onebitllms | BitNetLinear with STE + per-block scaling |
| train | trl SFTTrainer | standard HF training loop on GPU |
| quantize to 1-bit | onebitllms | freeze ternary weights |
| pack for CPU inference | smelt.quantize | TL1 packing |

Requires: `pip install onebitllms trl datasets`

In [1]:
import os
import sys

if "COLAB_GPU" in os.environ:
    os.system("pip install -q git+https://github.com/PritRaj1/smelt.git")
    os.system("pip install -q 'transformers>=4.51,<5' triton onebitllms trl datasets")
else:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Train with onebitllms

In [2]:
from onebitllms import replace_linear_with_bitnet_linear

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")

model = replace_linear_with_bitnet_linear(model)
print("BitNetLinear layers inserted")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

BitNetLinear layers inserted


In [3]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="/tmp/smelt_qat",
        num_train_epochs=1,
        learning_rate=1e-4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        logging_steps=10,
        bf16=True,
        max_length=256,
    ),
)
trainer.train()
trainer.save_model("/tmp/smelt_qat")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,12.542800
20,8.211700
30,7.590800
40,7.416400
50,6.978700
60,7.230200
70,7.111100
80,6.974300
90,6.820100
100,6.906600


wandb: WARNING URL not available in offline run


In [4]:
from onebitllms import quantize_to_1bit

quantize_to_1bit("/tmp/smelt_qat", "/tmp/smelt_qat_1bit")
print("quantized to 1-bit")

`torch_dtype` is deprecated! Use `dtype` instead!


quantized to 1-bit


## Pack for smelt inference

In [ ]:
import logging

import smelt

logging.basicConfig(level=logging.INFO)

model = AutoModelForCausalLM.from_pretrained(
    "/tmp/smelt_qat_1bit", dtype=torch.float32, device_map="cpu"
)
model.eval()
smelt.quantize(model)
model.generation_config.cache_implementation = "static"
model.forward = torch.compile(model.forward, fullgraph=True)
print("packed for CPU inference")

## Generate

In [ ]:
model = model.cpu()

prompts = [
    "The capital of UK is",
    "def fibonacci(n):",
    "Explain x86 AVX2 in simple terms:",
]

for p in prompts:
    ids = tokenizer.encode(p, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=32, do_sample=False)
    print(tokenizer.decode(out[0], skip_special_tokens=True))
    print()